# 03 · Attention from scratch

After this notebook you can derive and code scaled dot-product attention in NumPy (Q, K, V, the √d scaling, softmax, the causal mask), extend it to multi-head attention, prove your code matches PyTorch's `scaled_dot_product_attention` and `nn.MultiheadAttention`, and explain why attention is O(n²) and what MQA/GQA and FlashAttention do about it.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min · **Needs:** [02 · Embeddings](02_embeddings_and_similarity.ipynb), [NumPy for AI engineers](../04_numpy_pandas_pytorch/01_numpy_for_ai_engineers.ipynb), [embeddings, RNNs to attention](../06_deep_learning/05_embeddings_rnn_to_attention.ipynb)

## Why this matters in interviews

- "Implement self-attention" is the single most common from-scratch coding question for LLM roles; interviewers watch your shapes, the scaling and the mask.
- Every follow-up about cost (long context, KV cache size, FlashAttention, GQA) starts from the n x n score matrix you build here.
- Reading real model configs ("why is `k_proj` 1024 wide when `q_proj` is 3072?") needs MHA vs GQA vs MQA.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `tr03` | What is self-attention, in your own words? |
| `tr35` | How would you explain attention to someone with no maths background, and then formalise it? |
| `tr05` | Why multi-head attention instead of one big head? |
| `tr06` | What is the causal mask and why is it essential? |
| `tr11` | Why is attention O(n²), and what have people done about it? |
| `tr19` | What is FlashAttention and why did it matter so much? |
| `tr10` | What is grouped-query attention (GQA) and multi-query attention (MQA), and why do they exist? |
| `tr39` | Why are k_proj and v_proj 1024 wide when q_proj is 3072? |
| `tr20` | What is cross-attention and where do you see it? |

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

rng = np.random.default_rng(0)
torch.manual_seed(0)
np.set_printoptions(precision=3, suppress=True)
print("torch", torch.__version__)

## 1. Attention in plain English

**In plain English:** every token looks at the other tokens and decides how much each one matters *to it*, then builds its new representation as a weighted blend of them. In "the cat sat because **it** was tired", the token *it* should mostly look at *cat*.

Each token produces three vectors with three learned matrices:

| Vector | Role | Library analogy |
|---|---|---|
| **Query** q = x·W_q | "what am I looking for?" | the search terms |
| **Key** k = x·W_k | "what do I contain?" | the index / catalogue entry |
| **Value** v = x·W_v | "what do I hand over if chosen?" | the book itself |

It is a *soft* dictionary lookup: instead of returning the one value whose key matches, return a blend of all values, weighted by how well each key matches the query:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

## 2. Scaled dot-product attention, step by step

Six tokens, model width 8, head width 4, random weights - we only care about the mechanics and shapes here.

In [ ]:
tokens = ["the", "cat", "sat", "on", "the", "mat"]
T, d_model, d_k = len(tokens), 8, 4
X = rng.normal(size=(T, d_model))                 # one embedding row per token
W_q, W_k, W_v = (rng.normal(size=(d_model, d_k)) / np.sqrt(d_model) for _ in range(3))

Q, K, V = X @ W_q, X @ W_k, X @ W_v               # step 1: project
scores = Q @ K.T                                  # step 2: every query dotted with every key
scaled = scores / np.sqrt(d_k)                    # step 3: scale
print(f"X {X.shape} -> Q {Q.shape}, K {K.shape}, V {V.shape}")
print(f"scores = Q @ K.T: {scores.shape}  (row i = how well token i's query matches every key)")

In [ ]:
def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)       # subtract the max: same result, no overflow
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

weights = softmax(scaled)                         # step 4: each row becomes a probability distribution
out = weights @ V                                 # step 5: weighted sum of values
print("attention weights (rows = queries, cols = keys):\n", weights)
print(f"output {out.shape}: one new {d_k}-dim vector per token")
assert np.allclose(weights.sum(axis=1), 1.0), "every row of attention weights sums to 1"
assert (weights > 0).all(), "softmax never gives exactly zero - only a mask does"

naive = lambda z: np.exp(z) / np.exp(z).sum()
with np.errstate(over="ignore", invalid="ignore"):
    print("\nnaive softmax of [1000, 1001]:", naive(np.array([1000.0, 1001.0])), "  stable:", softmax(np.array([1000.0, 1001.0])))

The naive softmax overflows (`exp(1000) = inf`, giving `nan`); subtracting the row max first is mathematically identical and always safe. Interviewers check for this.

### Why divide by √d_k?

If the components of q and k are independent with variance 1, then q·k is a sum of d_k such products, so its **variance is d_k**. With d_k = 128, raw scores have a standard deviation of ~11; softmax of such large numbers is almost one-hot, and its gradient is almost zero, so training stalls. Dividing by √d_k brings the variance back to 1 regardless of head size.

In [ ]:
dims = [4, 16, 64, 256, 1024]
raw_std, scaled_std, max_w_raw, max_w_scaled = [], [], [], []
for d in dims:
    q = rng.normal(size=(2000, d)); k = rng.normal(size=(2000, d))
    dots = (q * k).sum(axis=1)
    raw_std.append(dots.std()); scaled_std.append((dots / np.sqrt(d)).std())
    rows = rng.normal(size=(500, 16, d)) @ rng.normal(size=d)       # 500 rows of 16 scores each
    max_w_raw.append(softmax(rows).max(axis=1).mean())
    max_w_scaled.append(softmax(rows / np.sqrt(d)).max(axis=1).mean())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(dims, raw_std, "o-", label="q·k (raw)")
axes[0].plot(dims, np.sqrt(dims), "k:", label="√d (theory)")
axes[0].plot(dims, scaled_std, "s-", label="q·k / √d")
axes[0].set_xscale("log", base=2); axes[0].set_yscale("log")
axes[0].set_xlabel("head dimension d_k"); axes[0].set_ylabel("std of attention score")
axes[0].set_title("Raw scores grow like √d; scaled scores stay at 1"); axes[0].legend()
axes[1].plot(dims, max_w_raw, "o-", label="softmax(q·k)")
axes[1].plot(dims, max_w_scaled, "s-", label="softmax(q·k / √d)")
axes[1].axhline(1 / 16, color="gray", ls=":", label="uniform over 16 keys")
axes[1].set_xscale("log", base=2); axes[1].set_ylim(0, 1.05)
axes[1].set_xlabel("head dimension d_k"); axes[1].set_ylabel("mean largest weight in a row")
axes[1].set_title("Without scaling, softmax saturates to one-hot"); axes[1].legend()
plt.tight_layout(); plt.show()

i64 = dims.index(64)
print(f"d_k=64: raw std {raw_std[i64]:.2f} (theory {np.sqrt(64):.0f}), scaled std {scaled_std[i64]:.2f}")
assert abs(raw_std[i64] / 8 - 1) < 0.1 and abs(scaled_std[i64] - 1) < 0.1
assert max_w_raw[-1] > 0.9 and max_w_scaled[-1] < 0.6

Left: the measured spread of raw scores follows √d exactly; the scaled scores stay at ~1. Right: without scaling, a row of 16 scores puts nearly all its weight on one key at large d (vanishing gradients for everything else); with scaling the distribution stays soft and trainable.

## 3. The causal mask

A decoder (GPT-style) model is trained to predict token t+1 from tokens 1..t. If position t could attend to t+1, it could simply copy the answer - so we set the scores for "future" keys to -∞ *before* the softmax, which makes their weight exactly 0. The same mask is what makes the KV cache possible later: past tokens never change.

To get a *readable* heatmap we hand-set the weights so that pronouns look for animals, verbs look for their subject, and the adjective looks for what it describes. A trained model learns patterns like these by itself.

In [ ]:
sent = ["the", "cat", "sat", "on", "the", "mat", "because", "it", "was", "tired"]
FEATS = ["animal", "place", "verb", "pronoun", "function", "adjective"]
kind = {"the": "function", "cat": "animal", "sat": "verb", "on": "function", "mat": "place",
        "because": "function", "it": "pronoun", "was": "verb", "tired": "adjective"}
F_ = {f: i for i, f in enumerate(FEATS)}
Xs = np.array([np.eye(len(FEATS))[F_[kind[w]]] for w in sent]) + 0.05 * rng.normal(size=(len(sent), len(FEATS)))

Wq_s = np.eye(len(FEATS)) * 1.0             # by default a token looks for its own kind ...
Wq_s[F_["pronoun"], F_["pronoun"]] = 0
Wq_s[F_["pronoun"], F_["animal"]] = 4.0     # ... but a pronoun looks for an animal
Wq_s[F_["verb"], F_["verb"]] = 0
Wq_s[F_["verb"], F_["animal"]] = 3.0        # a verb looks for its (animal) subject
Wq_s[F_["adjective"], F_["adjective"]] = 0
Wq_s[F_["adjective"], F_["pronoun"]] = 3.0  # an adjective looks for the pronoun it describes
Wk_s = np.eye(len(FEATS)) * 2.0             # a key advertises the token's own kind

def attention(Xin, Wq, Wk, Wv=None, causal=True):
    Qm, Km = Xin @ Wq, Xin @ Wk
    Vm = Xin if Wv is None else Xin @ Wv
    s = Qm @ Km.T / np.sqrt(Km.shape[1])
    if causal:
        s = np.where(np.triu(np.ones_like(s, dtype=bool), k=1), -np.inf, s)
    w = softmax(s)
    return w @ Vm, w

_, w_causal = attention(Xs, Wq_s, Wk_s, causal=True)
_, w_bidir = attention(Xs, Wq_s, Wk_s, causal=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, w, title in [(axes[0], w_causal, "Causal (decoder, GPT): no looking ahead"),
                     (axes[1], w_bidir, "Bidirectional (encoder, BERT): sees everything")]:
    im = ax.imshow(w, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(sent)), sent, rotation=60, fontsize=8)
    ax.set_yticks(range(len(sent)), sent, fontsize=8)
    ax.set_xlabel("key (attended to)"); ax.set_ylabel("query (attending)")
    ax.set_title(title, fontsize=10)
fig.colorbar(im, ax=axes, shrink=0.8, label="attention weight")
plt.show()

it, cat = sent.index("it"), sent.index("cat")
assert np.allclose(np.triu(w_causal, k=1), 0), "causal mask: zero weight on every future token"
assert w_causal[it].argmax() == cat, "'it' attends most to 'cat'"
assert np.allclose(w_causal.sum(axis=1), 1)
print(f"'it' -> 'cat' weight: {w_causal[it, cat]:.2f};  first row (causal) attends only to itself: {w_causal[0, 0]:.2f}")

Left: the upper triangle is exactly zero, so row 0 (`the`) can only attend to itself, and `it` puts most of its weight on `cat`. Right: without the mask, early tokens also attend to later ones (the bidirectional attention of encoder models like BERT, used for embeddings and classification, not generation).

## 4. Multi-head attention

One head computes one weighting per token pair. **Multi-head** attention runs h smaller heads in parallel, each with its own W_q, W_k, W_v of width d_model/h, so different heads can track different relations (syntax, coreference, position) at the same total cost. The head outputs are concatenated and mixed by an output projection W_o.

In [ ]:
def multi_head_attention(Xb, Wq, Wk, Wv, Wo, n_heads, causal=True, verbose=False):
    """Xb: (B, T, d_model). W*: (d_model, d_model) in x @ W convention. Returns output and weights."""
    B, T_, d = Xb.shape
    dh = d // n_heads
    split = lambda M: M.reshape(B, T_, n_heads, dh).transpose(0, 2, 1, 3)       # (B, h, T, dh)
    Qh, Kh, Vh = split(Xb @ Wq), split(Xb @ Wk), split(Xb @ Wv)
    s = Qh @ Kh.transpose(0, 1, 3, 2) / np.sqrt(dh)                            # (B, h, T, T)
    if causal:
        s = np.where(np.triu(np.ones((T_, T_), dtype=bool), k=1), -np.inf, s)
    w = softmax(s)
    heads = w @ Vh                                                             # (B, h, T, dh)
    concat = heads.transpose(0, 2, 1, 3).reshape(B, T_, d)                     # (B, T, d_model)
    out = concat @ Wo
    if verbose:
        for name, a in [("X", Xb), ("Q per head", Qh), ("scores", s), ("weights", w),
                        ("heads", heads), ("concat", concat), ("output", out)]:
            print(f"{name:11s} {str(a.shape):16s}")
    return out, w

B, T, d_model, n_heads = 2, 6, 16, 4
Xb = rng.normal(size=(B, T, d_model))
Wq, Wk, Wv, Wo = (rng.normal(size=(d_model, d_model)) / np.sqrt(d_model) for _ in range(4))
out_np, w_np = multi_head_attention(Xb, Wq, Wk, Wv, Wo, n_heads, verbose=True)

Shapes to memorise: split `(B, T, d)` into `(B, h, T, d/h)`, scores are `(B, h, T, T)`, concat back to `(B, T, d)`. Four heads of width 4 cost the same as one head of width 16; the gain is four *different* attention patterns.

## 5. Verify against PyTorch

Our NumPy code must give the same numbers as PyTorch's fused kernel `F.scaled_dot_product_attention` and the `nn.MultiheadAttention` layer when given the same weights. (PyTorch layers compute `x @ W.T`, so we pass transposed matrices.)

In [ ]:
tX = torch.tensor(Xb)
split_t = lambda M: M.view(B, T, n_heads, d_model // n_heads).transpose(1, 2)
q_t, k_t, v_t = (split_t(tX @ torch.tensor(W)) for W in (Wq, Wk, Wv))
sdpa = F.scaled_dot_product_attention(q_t, k_t, v_t, is_causal=True)          # (B, h, T, dh)
sdpa_out = sdpa.transpose(1, 2).reshape(B, T, d_model) @ torch.tensor(Wo)
assert np.allclose(sdpa_out.numpy(), out_np, atol=1e-10)
print("NumPy multi-head attention == F.scaled_dot_product_attention (is_causal=True)")

future = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)            # True above the diagonal
# Gotcha: SDPA's boolean attn_mask means True = ALLOWED, so pass the *allowed* positions (~future) ...
assert torch.allclose(F.scaled_dot_product_attention(q_t, k_t, v_t, attn_mask=~future), sdpa)

mha = nn.MultiheadAttention(d_model, n_heads, bias=False, batch_first=True, dtype=torch.float64)
with torch.no_grad():
    mha.in_proj_weight.copy_(torch.tensor(np.concatenate([Wq.T, Wk.T, Wv.T], axis=0)))
    mha.out_proj.weight.copy_(torch.tensor(Wo.T))
# ... while nn.MultiheadAttention's boolean attn_mask means True = BLOCKED, so pass `future` itself.
mha_out, mha_w = mha(tX, tX, tX, attn_mask=future, need_weights=True, average_attn_weights=False)
assert np.allclose(mha_out.detach().numpy(), out_np, atol=1e-10)
assert np.allclose(mha_w.detach().numpy(), w_np, atol=1e-10)
print("NumPy multi-head attention == nn.MultiheadAttention (outputs and per-head weights)")
print(f"max abs difference: {np.abs(mha_out.detach().numpy() - out_np).max():.1e}")

Identical to ~1e-15 (float64 rounding). Now you know exactly what those library calls compute.

## 6. The O(n²) problem

The score matrix has one entry per (query, key) pair: n² per head per layer. Double the context and the attention work (and, naively, the memory) quadruples.

In [ ]:
def attn_single(Qm, Km, Vm):
    return softmax(Qm @ Km.T / np.sqrt(Qm.shape[1])) @ Vm

def best_time(fn, repeats=3):
    """Fastest of a few runs: robust to a busy machine."""
    best = float("inf")
    for _ in range(repeats):
        t0 = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t0)
    return best

lengths = [256, 512, 1024, 2048]
times = []
for n in lengths:
    Qn, Kn, Vn = (rng.normal(size=(n, 64)).astype(np.float32) for _ in range(3))
    times.append(best_time(lambda: attn_single(Qn, Kn, Vn)))

seq = np.array([1_024, 4_096, 32_768, 131_072])
score_bytes = seq.astype(float) ** 2 * 32 * 2            # 32 heads, fp16, one layer, one sequence
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].loglog(lengths, np.array(times) * 1e3, "o-", label="measured (NumPy, 1 head, d=64)")
axes[0].loglog(lengths, times[0] * 1e3 * (np.array(lengths) / lengths[0]) ** 2, "k:", label="∝ n²")
axes[0].set_xticks(lengths, [str(n) for n in lengths]); axes[0].xaxis.set_minor_locator(plt.NullLocator())
axes[0].set_xlabel("sequence length n"); axes[0].set_ylabel("ms"); axes[0].legend()
axes[0].set_title("Attention time grows ~quadratically")
axes[1].loglog(seq, score_bytes / 2**30, "s-", color="tab:red")
for n, b in zip(seq, score_bytes):
    label = f"{b / 2**30:,.2f} GiB" if b < 2**40 else f"{b / 2**40:.1f} TiB"
    axes[1].annotate(label, (n, b / 2**30), xytext=(-8, 6), textcoords="offset points", ha="right", fontsize=8)
axes[1].set_xticks(seq, ["1k", "4k", "32k", "128k"]); axes[1].xaxis.set_minor_locator(plt.NullLocator())
axes[1].set_xlabel("sequence length n"); axes[1].set_ylabel("GiB")
axes[1].set_title("Full score matrix: 32 heads, fp16, ONE layer")
plt.tight_layout(); plt.show()
print("time ratio 2048 vs 256 tokens:", round(times[-1] / times[0], 1), "(linear would be 8, quadratic 64)")
assert times[-1] / times[0] > 12

At 128k tokens a single layer's score matrix would be ~1 TiB in fp16 - impossible to materialise. What people did about it:

| Approach | Idea | Exact? |
|---|---|---|
| **FlashAttention** (v1-v3) | compute attention tile by tile in fast on-chip SRAM with an *online softmax*; never write the n x n matrix to GPU memory. Same FLOPs, far less memory traffic -> 2-4x faster, O(n) memory | yes |
| **Sparse / sliding-window** (Longformer, Mistral's SWA) | each token attends to a local window (+ a few global tokens) | no |
| **Linear attention / state-space models** (Mamba) | replace softmax attention with a recurrence, O(n) | different model |
| **KV-cache tricks** (GQA, MQA, paging, quantisation) | shrink the *memory* of past keys and values | yes |

FlashAttention's key trick fits in a few lines: process keys in blocks, keep a running max and a running denominator, and rescale the partial output when the max changes. The result is *exactly* the same as full softmax attention:

In [ ]:
def tiled_attention(q_row, Km, Vm, block=64):
    """One query row, keys/values streamed in blocks (online softmax). Never holds all n scores at once."""
    m, l, acc = -np.inf, 0.0, np.zeros(Vm.shape[1])
    for start in range(0, len(Km), block):
        s = Km[start:start + block] @ q_row / np.sqrt(len(q_row))   # scores for this block only
        m_new = max(m, s.max())
        scale = np.exp(m - m_new)                                   # rescale what we had so far
        p = np.exp(s - m_new)
        l = l * scale + p.sum()
        acc = acc * scale + p @ Vm[start:start + block]
        m = m_new
    return acc / l

Qn, Kn, Vn = (rng.normal(size=(1000, 64)) for _ in range(3))
full = attn_single(Qn[:5], Kn, Vn)
tiled = np.stack([tiled_attention(Qn[i], Kn, Vn, block=64) for i in range(5)])
assert np.allclose(full, tiled)
print(f"tiled online-softmax attention == full attention (max diff {np.abs(full - tiled).max():.1e}), "
      f"holding 64 scores at a time instead of 1000")

## 7. MHA vs MQA vs GQA

During generation every past token's keys and values are cached (notebook 06). With standard multi-head attention (MHA) each of the h query heads has its own K and V head, so the cache is large. Two fixes share K/V heads between query heads:

- **MQA (multi-query):** all query heads share **one** K/V head. Smallest cache, some quality loss.
- **GQA (grouped-query):** query heads are split into g groups, each group shares one K/V head. Llama 2 70B, Llama 3, Mistral and Qwen use it: nearly MHA quality, a fraction of the cache.

In [ ]:
def draw_heads(ax, n_q, n_kv, title):
    for i in range(n_q):
        ax.add_patch(plt.Rectangle((i, 1.6), 0.8, 0.5, color="tab:blue"))
    group = n_q // n_kv
    for j in range(n_kv):
        x = j * group + (group - 1) / 2
        ax.add_patch(plt.Rectangle((x, 0), 0.8, 0.5, color="tab:orange"))
        for i in range(j * group, (j + 1) * group):
            ax.plot([i + 0.4, x + 0.4], [1.6, 0.5], color="gray", lw=0.8)
    ax.set_xlim(-0.3, n_q); ax.set_ylim(-0.2, 2.3); ax.axis("off"); ax.set_title(title, fontsize=10)

fig, axes = plt.subplots(1, 3, figsize=(11, 2.4))
draw_heads(axes[0], 8, 8, "MHA: 8 query heads, 8 K/V heads")
draw_heads(axes[1], 8, 2, "GQA: 8 query heads, 2 K/V heads")
draw_heads(axes[2], 8, 1, "MQA: 8 query heads, 1 K/V head")
fig.text(0.01, 0.75, "queries", color="tab:blue"); fig.text(0.01, 0.2, "keys/values", color="tab:orange")
plt.show()

GQA in code: compute K and V with fewer heads, then let each group of query heads use its shared K/V head. PyTorch's `scaled_dot_product_attention(..., enable_gqa=True)` does the same broadcast internally.

In [ ]:
def gqa_numpy(Qh, Kh, Vh, causal=True):
    """Qh: (B, h_q, T, dh); Kh, Vh: (B, h_kv, T, dh) with h_q divisible by h_kv."""
    group = Qh.shape[1] // Kh.shape[1]
    Kr, Vr = np.repeat(Kh, group, axis=1), np.repeat(Vh, group, axis=1)   # heads 0..g-1 share K/V head 0, ...
    s = Qh @ Kr.transpose(0, 1, 3, 2) / np.sqrt(Qh.shape[-1])
    if causal:
        s = np.where(np.triu(np.ones(s.shape[-2:], dtype=bool), k=1), -np.inf, s)
    return softmax(s) @ Vr

Bq, h_q, h_kv, Tq, dh = 2, 8, 2, 5, 16
Qh = rng.normal(size=(Bq, h_q, Tq, dh)); Kh = rng.normal(size=(Bq, h_kv, Tq, dh)); Vh = rng.normal(size=(Bq, h_kv, Tq, dh))
ours = gqa_numpy(Qh, Kh, Vh)
theirs = F.scaled_dot_product_attention(torch.tensor(Qh), torch.tensor(Kh), torch.tensor(Vh), is_causal=True, enable_gqa=True)
assert np.allclose(ours, theirs.numpy())
Kfull, Vfull = np.repeat(Kh, h_q // h_kv, axis=1), np.repeat(Vh, h_q // h_kv, axis=1)
assert np.allclose(gqa_numpy(Qh, Kfull, Vfull), ours), "GQA == MHA whose K/V heads are copies within each group"
print(f"GQA ({h_q} query heads, {h_kv} K/V heads) == torch SDPA(enable_gqa=True); output {ours.shape}")

What sharing buys, for one layer of a Llama-3-8B-sized model (d_model 4096, 32 query heads, head_dim 128), fp16 cache:

In [ ]:
d_model_8b, n_q_8b, head_dim, layers, bytes_fp16 = 4096, 32, 128, 32, 2
rows = []
for name, n_kv in [("MHA", 32), ("GQA-8 (Llama 3 8B)", 8), ("MQA", 1)]:
    kv_proj_params = 2 * d_model_8b * n_kv * head_dim                     # W_k and W_v
    kv_per_token = 2 * layers * n_kv * head_dim * bytes_fp16              # K and V, all layers
    rows.append((name, n_kv, kv_proj_params / 1e6, kv_per_token / 1024, kv_per_token * 131_072 / 2**30))
table = pd.DataFrame(rows, columns=["variant", "K/V heads", "W_k+W_v params/layer (M)",
                                    "KV cache per token (KiB)", "KV cache @128k tokens (GiB)"])
print(table.round(2).to_string(index=False))
assert table["KV cache per token (KiB)"].tolist() == [512.0, 128.0, 16.0]

GQA-8 cuts the cache 4x versus MHA (128 KiB instead of 512 KiB per token), which directly means 4x longer contexts or 4x bigger batches on the same GPU. That is also the answer to **tr39**: in Llama 3.2 3B, `q_proj` maps 3072 -> 3072 (24 query heads x 128) but `k_proj`/`v_proj` map 3072 -> **1024** (8 K/V heads x 128) - it is GQA with 3 query heads per K/V head.

## 8. Cross-attention (briefly)

In **self**-attention Q, K and V all come from the same sequence. In **cross**-attention the queries come from one sequence and the keys/values from *another*: the decoder of a translation model (or Whisper, T5) attends over the encoder's output; multimodal models attend from text to image features. There is no causal mask over the source, and the weight matrix is (target length x source length). Below, hand-set weights make each French word attend to the English word with the same meaning - note the crossing, because French puts the adjective after the noun.

In [ ]:
concepts = ["DET", "BLACK", "CAT"]
src, tgt = ["the", "black", "cat"], ["le", "chat", "noir"]
meaning = {"the": "DET", "black": "BLACK", "cat": "CAT", "le": "DET", "chat": "CAT", "noir": "BLACK"}
X_src = np.array([np.eye(3)[concepts.index(meaning[w])] for w in src]) + 0.05 * rng.normal(size=(3, 3))
X_tgt = np.array([np.eye(3)[concepts.index(meaning[w])] for w in tgt]) + 0.05 * rng.normal(size=(3, 3))
Wq_c = Wk_c = np.eye(3) * 3.0
w_cross = softmax((X_tgt @ Wq_c) @ (X_src @ Wk_c).T / np.sqrt(3))       # queries: French; keys: English

fig, ax = plt.subplots(figsize=(4.2, 3.4))
ax.imshow(w_cross, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(3), src); ax.set_yticks(range(3), tgt)
ax.set_xlabel("source (encoder, English) = keys/values"); ax.set_ylabel("target (decoder) = queries")
ax.set_title("Cross-attention alignment")
plt.show()
assert [src[i] for i in w_cross.argmax(axis=1)] == ["the", "cat", "black"]

## Try it yourself

**1. Padding mask.** Batch two sentences of different lengths: the second has only 4 real tokens and 2 padding tokens. Build a mask so that *no query attends to a padding key*, and assert the padded keys get exactly zero weight.

In [ ]:
# Solution — try it yourself first, then run this
lengths_b = np.array([6, 4])
Xp = rng.normal(size=(2, 6, 8)); Wqp, Wkp = rng.normal(size=(8, 4)), rng.normal(size=(8, 4))
s = (Xp @ Wqp) @ (Xp @ Wkp).transpose(0, 2, 1) / 2.0                     # (2, 6, 6)
is_pad_key = np.arange(6)[None, :] >= lengths_b[:, None]                 # (2, 6): True where the key is padding
s = np.where(is_pad_key[:, None, :], -np.inf, s)                         # broadcast over queries
w = softmax(s)
print("weights on the 2 padded keys of sentence 2:", w[1, :, 4:].max())
assert np.all(w[1, :, 4:] == 0) and np.allclose(w.sum(-1), 1)

**2. Why positional information is needed.** Show that attention *without* positions is permutation-equivariant: shuffle the input tokens, and the outputs are the same vectors, shuffled the same way (no causal mask). So "dog bites man" and "man bites dog" would look identical to it.

In [ ]:
# Solution — try it yourself first, then run this
perm = rng.permutation(len(Xs))
out_a, _ = attention(Xs, Wq_s, Wk_s, causal=False)
out_b, _ = attention(Xs[perm], Wq_s, Wk_s, causal=False)
assert np.allclose(out_a[perm], out_b)
print("attention(shuffled X) == shuffled attention(X): order is invisible without positional encodings")

**3. Read a real config.** Llama 3.2 3B: hidden size 3072, 24 query heads, 8 K/V heads, head_dim 128, 28 layers. Compute the attention parameters per layer (q, k, v, o projections, no biases) and the fp16 KV cache per token. What would they be with plain MHA?

In [ ]:
# Solution — try it yourself first, then run this
def attn_params(d, n_q, n_kv, hd):
    return d * n_q * hd + 2 * d * n_kv * hd + n_q * hd * d        # q + (k, v) + o

for label, n_kv in [("GQA (actual)", 8), ("MHA (hypothetical)", 24)]:
    p = attn_params(3072, 24, n_kv, 128)
    kv = 2 * 28 * n_kv * 128 * 2
    print(f"{label:20s} attention params/layer {p / 1e6:5.1f}M | KV cache per token {kv / 1024:.0f} KiB")
assert attn_params(3072, 24, 8, 128) == 3072 * 3072 * 2 + 2 * 3072 * 1024

## Say it in an interview

- **No-maths version:** "Each word asks a question, every word advertises what it contains, and each word rebuilds itself as a blend of the words whose adverts best answer its question."
- **Formal version:** Q = XW_q, K = XW_k, V = XW_v; weights = softmax(QKᵀ/√d_k + mask); output = weights·V; multi-head: h parallel heads of width d/h, concatenated, then W_o.
- **Why √d_k:** q·k has variance d_k; without scaling the softmax saturates and gradients vanish.
- **Causal mask:** -∞ above the diagonal before softmax, so position t never sees t+1; it is what lets the model train on every position in parallel and what makes the KV cache valid.
- **Why multi-head:** several different relations per layer (syntax, coreference, position) at the same cost as one wide head.
- **Cost:** O(n²·d) time, O(n²) naive memory per head per layer. FlashAttention keeps it exact but tiles it with an online softmax, so memory is O(n) and it is IO-efficient; sliding windows and SSMs change the model instead.
- **GQA/MQA:** share K/V heads across query heads to shrink the KV cache (Llama 3 8B: 32 query heads, 8 K/V heads -> 4x smaller cache); MQA is the extreme with one K/V head.
- **Traps:** forgetting to scale or to subtract the max in softmax; masking *after* softmax (weights no longer sum to 1); mixing up `(B, T, h, dh)` and `(B, h, T, dh)` when reshaping; mixing up mask conventions (a PyTorch boolean mask means True = *blocked* for `nn.MultiheadAttention` but True = *allowed* for `F.scaled_dot_product_attention` - both asserted above).

## Next

- [04 · Transformer block and a tiny GPT](04_transformer_block_and_tiny_gpt.ipynb) - wrap this attention into a block and train a model.
- [06 · KV cache and inference speed](06_kv_cache_and_inference_speed.ipynb) - why the causal mask makes caching K and V possible.
- [Implement-from-scratch drills](../17_interview_coding_rounds/07_implement_from_scratch_drills.ipynb) - timed practice of exactly this code.
- Theory: [genai_flow course](../../genai_flow/index.html) (chapter 4 "Attention", chapter 18b "Reading print(model)" for GQA in Llama 3.2) · [interview bank](../../ai_interview_prep/index.html)